In [ ]:
%pip install groq scikit-learn pandas mlflow
import os
from groq import Groq
from dotenv import load_dotenv
load_dotenv()
api_key = os.getenv("GROQ_API_KEY")
client = Groq(api_key=api_key)

  Using cached scikit_learn-1.7.2-cp313-cp313-win_amd64.whl.metadata (11 kB)
  Using cached pandas-2.3.3-cp313-cp313-win_amd64.whl.metadata (19 kB)
  Using cached mlflow-3.6.0-py3-none-any.whl.metadata (31 kB)
  Using cached numpy-2.3.5-cp313-cp313-win_amd64.whl.metadata (60 kB)
  Using cached scipy-1.16.3-cp313-cp313-win_amd64.whl.metadata (60 kB)
  Using cached joblib-1.5.2-py3-none-any.whl.metadata (5.6 kB)
  Using cached threadpoolctl-3.6.0-py3-none-any.whl.metadata (13 kB)
  Using cached pytz-2025.2-py2.py3-none-any.whl.metadata (22 kB)
  Using cached tzdata-2025.2-py2.py3-none-any.whl.metadata (1.4 kB)
  Using cached mlflow_skinny-3.6.0-py3-none-any.whl.metadata (31 kB)
  Using cached mlflow_tracing-3.6.0-py3-none-any.whl.metadata (19 kB)
  Using cached flask_cors-6.0.1-py3-none-any.whl.metadata (5.3 kB)
  Using cached flask-3.1.2-py3-none-any.whl.metadata (3.2 kB)
  Using cached alembic-1.17.2-py3-none-any.whl.metadata (7.2 kB)
  Using cached cryptography-46.0.3-cp311-abi3-win_a


[notice] A new release of pip is available: 24.3.1 -> 25.3
[notice] To update, run: python.exe -m pip install --upgrade pip


In [ ]:
# === Imports ===
import json, csv, time
import pandas as pd
from sklearn.metrics import accuracy_score, classification_report
import mlflow

In [ ]:
# === Config ===
MODEL = "llama-3.1-8b-instant"   # change if needed

EVAL_PATH = "G:\Resume-Matcher\data\eval.jsonl"
OUT_DIR = "results"
os.makedirs(OUT_DIR, exist_ok=True)
RESP_CSV = os.path.join(OUT_DIR, "responses.csv")
HUMAN_CSV = os.path.join(OUT_DIR, "human_rubric_template.csv")

<>:6: SyntaxWarning: invalid escape sequence '\R'
<>:6: SyntaxWarning: invalid escape sequence '\R'
C:\Users\SAHIL\AppData\Local\Temp\ipykernel_1356\2230405859.py:6: SyntaxWarning: invalid escape sequence '\R'
  EVAL_PATH = "G:\Resume-Matcher\data\eval.jsonl"


In [7]:
# === Helper: load eval items ===
def load_eval(path):
    items = []
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            if not line.strip(): continue
            items.append(json.loads(line))
    return items

# === Load prompt templates (fallbacks if files missing) ===
def load_text(path, fallback):
    try:
        with open(path, "r", encoding="utf-8") as f:
            return f.read()
    except Exception:
        return fallback

zero_template = load_text("/mnt/data/zero_shot.txt",
                          "Decide match: Given Candidate and Job, answer with label (High/Medium/Low) and 1-line justification.")
few_template_text = load_text("/mnt/data/few_shot_k3.txt",
                              None)  # we'll use embedded examples if missing
cot_template = load_text("/mnt/data/cot_meta.txt",
                         "Think step-by-step, then give label (High/Medium/Low) and short justification.")

# === Few-shot examples (k=3) used inline if file not structured ===
few_shots_k3 = [
    {"role":"user","content":"Q: Candidate: 'Experienced Python dev with ML background' Job: 'ML Engineer' -> Match? Provide justification."},
    {"role":"assistant","content":"High — Python + ML experience directly aligns with ML Engineer duties."},
    {"role":"user","content":"Q: Candidate: 'Frontend React developer' Job: 'Backend Node.js Developer' -> Match? Provide justification."},
    {"role":"assistant","content":"Low — lacks backend/Node.js experience required for the role."},
    {"role":"user","content":"Q: Candidate: 'Data analyst with SQL and Tableau' Job: 'Business Analyst' -> Match? Provide justification."},
    {"role":"assistant","content":"Medium — analytics skills align but stakeholder experience unclear."}
]

# === Call function for the API ===
def run_chat(messages):
    resp = client.chat.completions.create(model=MODEL, messages=messages)
    # message object -> attribute .content
    text = resp.choices[0].message.content
    time.sleep(0.2)
    return text.strip()

# === Build messages per strategy ===
def build_zero_messages(candidate, job):
    content = f"{zero_template}\n\nCandidate: {candidate}\nJob: {job}\n\nAnswer:"
    return [{"role":"system","content":"You are a concise resume-job matcher."},
            {"role":"user","content":content}]

def build_few_messages(candidate, job):
    messages = [{"role":"system","content":"You are a concise resume-job matcher."}]
    # if file provided and seems to contain examples, add as a single user message; else add structured few_shots_k3
    if few_template_text and len(few_template_text) > 50:
        messages.append({"role":"user","content": few_template_text})
    else:
        messages += few_shots_k3
    messages.append({"role":"user","content": f"Q: Candidate: '{candidate}' Job: '{job}' ->"})
    return messages

def build_cot_messages(candidate, job):
    content = f"{cot_template}\n\nCandidate: {candidate}\nJob: {job}\n\nNow think step-by-step and give label+1-line justification:"
    return [{"role":"system","content":"You are a helpful, thoughtful resume-job assessor."},
            {"role":"user","content":content}]

# === Simple label extractor ===
def extract_label(text):
    t = text.lower()
    if "high" in t: return "High"
    if "medium" in t: return "Medium"
    if "low" in t: return "Low"
    # fallback: map words
    if "strong" in t or "good" in t or "yes" in t: return "High"
    if "partial" in t or "some" in t: return "Medium"
    return "Unknown"

# === Main eval loop ===
items = load_eval(EVAL_PATH)
rows = []
for it in items:
    cid = it.get("id")
    cand = it.get("candidate")
    job = it.get("job")
    gt = it.get("ground_truth")
    # run strategies
    z = run_chat(build_zero_messages(cand, job))
    f = run_chat(build_few_messages(cand, job))
    a = run_chat(build_cot_messages(cand, job))
    z_label = extract_label(z)
    f_label = extract_label(f)
    a_label = extract_label(a)
    rows.append({
        "id": cid, "candidate": cand, "job": job, "ground_truth": gt,
        "zero_text": z, "few_text": f, "adv_text": a,
        "zero_label": z_label, "few_label": f_label, "adv_label": a_label
    })
    print(f"#{cid} done → zero:{z_label} few:{f_label} adv:{a_label}")

#1 done → zero:Medium few:High adv:High
#2 done → zero:Medium few:Low adv:Medium
#3 done → zero:Medium few:High adv:High
#4 done → zero:Medium few:Low adv:Medium
#5 done → zero:Medium few:High adv:Medium
#6 done → zero:Medium few:Low adv:High
#7 done → zero:Medium few:High adv:High
#8 done → zero:Medium few:Low adv:High
#9 done → zero:Medium few:High adv:High
#10 done → zero:High few:High adv:High
#11 done → zero:Medium few:Low adv:High
#12 done → zero:High few:High adv:High
#13 done → zero:Medium few:Medium adv:High
#14 done → zero:High few:High adv:High
#15 done → zero:High few:High adv:High
#16 done → zero:Medium few:Low adv:High
#17 done → zero:Medium few:High adv:Medium
#18 done → zero:Medium few:Medium adv:Medium
#19 done → zero:High few:High adv:High
#20 done → zero:High few:High adv:High


In [8]:
# === Save responses CSV ===
df = pd.DataFrame(rows)
df.to_csv(RESP_CSV, index=False)

# === Create human rubric template ===
# Columns: id, strategy, response_text, teammate_rating(1-5), comments
hum_rows = []
for r in rows:
    for strat,txt in [("zero", r["zero_text"]), ("few", r["few_text"]), ("adv", r["adv_text"])]:
        hum_rows.append({
            "id": r["id"],
            "strategy": strat,
            "response_text": txt,
            "factuality(1-5)": "",
            "helpfulness(1-5)": "",
            "clarity(1-5)": "",
            "comments": ""
        })
pd.DataFrame(hum_rows).to_csv(HUMAN_CSV, index=False)

# === Quantitative eval (accuracy + classification report) ===
def compute_metrics(pred_col):
    y_true = [r["ground_truth"] for r in rows]
    y_pred = [r[pred_col] for r in rows]
    acc = accuracy_score(y_true, y_pred)
    report = classification_report(y_true, y_pred, zero_division=0)
    return acc, report

for col, name in [("zero_label","Zero-shot"), ("few_label","Few-shot"), ("adv_label","CoT/Meta")]:
    acc, rep = compute_metrics(col)
    print(f"\n=== {name} — accuracy: {acc:.3f} ===\n{rep}")

# === Log to MLflow (simple) ===
mlflow.set_experiment("prompt_eval")
with mlflow.start_run(run_name="prompt_strategies"):
    for col,name in [("zero_label","Zero-shot"),("few_label","Few-shot"),("adv_label","CoT/Meta")]:
        acc, _ = compute_metrics(col)
        mlflow.log_metric(f"{name}_accuracy", acc)
    mlflow.log_artifact(RESP_CSV)
    mlflow.log_artifact(HUMAN_CSV)

print("Done. Responses saved to:", RESP_CSV, "Human rubric template:", HUMAN_CSV)

c:\Users\SAHIL\AppData\Local\Programs\Python\Python313\Lib\site-packages\mlflow\tracking\_tracking_service\utils.py:140: FutureWarning: Filesystem tracking backend (e.g., './mlruns') is deprecated. Please switch to a database backend (e.g., 'sqlite:///mlflow.db'). For feedback, see: https://github.com/mlflow/mlflow/issues/18534
  return FileStore(store_uri, store_uri)
2025/11/29 02:02:45 INFO mlflow.tracking.fluent: Experiment with name 'prompt_eval' does not exist. Creating a new experiment.



=== Zero-shot — accuracy: 0.600 ===
              precision    recall  f1-score   support

        High       1.00      0.55      0.71        11
         Low       0.00      0.00      0.00         3
      Medium       0.43      1.00      0.60         6

    accuracy                           0.60        20
   macro avg       0.48      0.52      0.44        20
weighted avg       0.68      0.60      0.57        20


=== Few-shot — accuracy: 0.750 ===
              precision    recall  f1-score   support

        High       0.83      0.91      0.87        11
         Low       0.50      1.00      0.67         3
      Medium       1.00      0.33      0.50         6

    accuracy                           0.75        20
   macro avg       0.78      0.75      0.68        20
weighted avg       0.83      0.75      0.73        20


=== CoT/Meta — accuracy: 0.550 ===
              precision    recall  f1-score   support

        High       0.60      0.82      0.69        11
         Low       0